# AASIST - *Hugging Face*
## Basic implementation
Pre-trained model from [Hugging Face](https://huggingface.co/MTUCI/AASIST3)

### Objective
The primary objective of **AASIST3** is to provide a robust defense against speech deepfakes and synthetic audio attacks. It is designed specifically for the **ASVspoof 2024 Challenge**, aiming to distinguish between "bonafide" (human) speech and "spoof" (AI-generated or replayed) audio across diverse languages and recording conditions.

### Architecture Description
AASIST3 evolves the original AASIST (*Anti-spoofing with Adaptive Softmax and Instance-wise Temperature*) framework by integrating **Kolmogorov-Arnold Networks (KAN)** and Self-Supervised Learning (SSL) features.


The architecture is structured into the following functional stages:

1.  **SSL Feature Extraction:**
    The model utilizes a **Wav2Vec2** encoder as a front-end to extract high-level representations from raw audio waveforms. This allows the model to benefit from robust features learned during large-scale self-supervised pre-training.

2.  **KAN Bridge & Transformation:**
    Unlike traditional architectures that rely on standard MLP/Linear layers, AASIST3 incorporates **KAN Linear Layers**. These layers use learnable activation functions on the edges (splines) rather than fixed activations on nodes, allowing for more complex and efficient feature transformation.

3.  **Residual Encoding:**
    The extracted features pass through a series of **Residual Blocks** to capture hierarchical spectral and temporal patterns, ensuring stable gradient flow during training.

4.  **Graph Attention Networks (GAT):**
    To model the relationship between different segments of the audio signal, the architecture employs two specialized graph modules:
    * **GAT-S (Spatial):** Focuses on modeling dependencies across different frequency bins or feature dimensions.
    * **GAT-T (Temporal):** Focuses on modeling the long-term temporal dependencies of the speech signal.

5.  **Multi-branch Inference & Output:**
    The model uses four parallel inference branches integrated with **master tokens** to aggregate global information. The final classification (bonafide vs. spoof) is performed by an output layer also powered by KAN, which provides the final decision logic.

## Repo cloning and model import

In [1]:
!git clone https://github.com/mtuciru/AASIST3.git
!cd AASIST3

Cloning into 'AASIST3'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 69 (delta 22), reused 69 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 36.43 KiB | 4.05 MiB/s, done.
Resolving deltas: 100% (22/22), done.


### Dependencies installation

In [ ]:
#!pip install -r '/content/AASIST3/requirements.txt'

In [2]:
# 1. Purge all potentially conflicting packages
!pip uninstall -y torch torchvision torchaudio torchcodec datasets

# 2. Install the strictly aligned PyTorch ecosystem (CUDA 12.1)
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# 3. Install remaining dependencies, pinning datasets to the stable 2.x branch
!pip install transformers accelerate datasets==2.19.1 soundfile

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu
Found existing installation: torchcodec 0.10.0
Uninstalling torchcodec-0.10.0:
  Successfully uninstalled torchcodec-0.10.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 540.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 78.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 66.9 MB/

> **RuntimeError:** Could not load libtorchcodec
This error is a binary linkage failure. It occurs when the Python wrapper for torchcodec cannot initialize its underlying C++ engine.

[Post adressing this issue](https://discuss.huggingface.co/t/issue-with-torchcodec-when-fine-tuning-whisper-asr-model/169315/2)

In [ ]:
# Colab VM or Linux
#!apt-get update && apt-get install -y ffmpeg
#!pip install -U "datasets[audio]" "torch==2.8.*" "torchcodec==0.7.*"
# HF docs: audio decoding uses TorchCodec + FFmpeg
# https://huggingface.co/docs/datasets/en/audio_load

### Import testing

In [3]:
import sys

# Add the repository root to the search path
repo_root = "/content/AASIST3/"
if repo_root not in sys.path:
    sys.path.append(repo_root)

In [4]:
import os
os.environ["TORCHAUDIO_USE_TORCHCODEC"] = "0"

In [5]:
from model import aasist3

# Load the model from Hugging Face Hub
model = aasist3.from_pretrained("MTUCI/AASIST3")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/550 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.29G [00:00<?, ?B/s]

#### Architecture

In [ ]:
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/550 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.29G [00:00<?, ?B/s]

aasist3(
  (w2v_encoder): Wav2Vec2Encoder(
    (model): Wav2Vec2Model(
      (feature_extractor): Wav2Vec2FeatureEncoder(
        (conv_layers): ModuleList(
          (0): Wav2Vec2LayerNormConvLayer(
            (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
            (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (activation): GELUActivation()
          )
          (1-4): 4 x Wav2Vec2LayerNormConvLayer(
            (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
            (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (activation): GELUActivation()
          )
          (5-6): 2 x Wav2Vec2LayerNormConvLayer(
            (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
            (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (activation): GELUActivation()
          )
        )
      )
      (feature_projection): Wav2Vec2FeatureProjection(
        (layer_norm):

---

### Inference Testing

In [6]:
import torch
import torch.nn.functional as F
from datasets import load_dataset

# 1. Load the dataset in streaming mode
# Using ASVspoof 2019 LA as it is a standard benchmark
ds = load_dataset("Bisher/ASVspoof_2019_LA", split="test", streaming=True)

# 2. Fetch the first sample
#   2.1. Get the sample
sample = next(iter(ds))
#   2.2. Extract Audio
audio_data = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]
#   2.3. Map the Label (Crucial for evaluation)
#   The 'key' contains 'bonafide' or 'spoof'
is_spoof = sample["key"] == "spoof"
label = 1 if is_spoof else 0

print(f"Speaker: {sample['speaker_id']}")
print(f"System ID: {sample['system_id']} (Attack type)")

# 3. Convert to Torch Tensor and ensure Float32
audio = torch.from_numpy(audio_data).float().unsqueeze(0)

print(f"Loaded sample with Label: {label} ({'Spoof' if label == 1 else 'Bonafide'})")
print(f"Sample Rate: {sr}Hz, Shape: {audio.shape}")

Speaker: LA_0039
System ID: A11 (Attack type)
Loaded sample with Label: 0 (Bonafide)
Sample Rate: 16000Hz, Shape: torch.Size([1, 22831])


In [7]:
# 1. Get the sample
sample = next(iter(ds))

# 2. Extract Audio
audio_data = sample["audio"]["array"]
sr = sample["audio"]["sampling_rate"]

# 3. Map the Label (Crucial for evaluation)
# The 'key' contains 'bonafide' or 'spoof'
is_spoof = sample["key"] == "spoof"
label = 1 if is_spoof else 0

print(f"Speaker: {sample['speaker_id']}")
print(f"System ID: {sample['system_id']} (Attack type)")
print(f"Mapped Label: {label} ({'Spoof' if label == 1 else 'Bonafide'})")

Speaker: LA_0039
System ID: A11 (Attack type)
Mapped Label: 0 (Bonafide)


In [8]:
# A. Resampling to 16kHz
if sr != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
    audio = resampler(audio)

# B. Mono Check
if audio.shape[0] > 1:
    audio = torch.mean(audio, dim=0, keepdim=True)

# C. Pre-emphasis Filter
# y[n] = x[n] - 0.97 * x[n-1]
# Suppresses spectral tilt and boosts high-freq artifacts
audio = torch.cat((audio[:, :1], audio[:, 1:] - 0.97 * audio[:, :-1]), dim=1)

# D. Z-Score Normalization (Standardization)
# Centers the distribution for the transformer encoder
audio = (audio - audio.mean()) / (audio.std() + 1e-7)

# E. Temporal Shaping (Padding/Truncating)
# Model expects exactly 64,600 samples (~4.03 seconds)
target_len = 64600
current_len = audio.shape[1]

if current_len < target_len:
    audio = F.pad(audio, (0, target_len - current_len))
else:
    audio = audio[:, :target_len]

In [9]:
# Move model to eval mode and detect device
model.eval()
device = next(model.parameters()).device

with torch.no_grad():
    # Model expects input shape (Batch, Time) -> (1, 64600)
    input_tensor = audio.to(device)

    # Run forward pass
    output = model(input_tensor)

    # Calculate probabilities and predictions
    probabilities = torch.softmax(output, dim=1)
    prediction = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][prediction].item()

    # Results
    print("--- Inference Results ---")
    print(f"Ground Truth: {'Spoof' if label == 1 else 'Bonafide'}")
    print(f"Prediction: {'Spoof' if prediction == 1 else 'Bonafide'}")
    print(f"Confidence: {confidence:.2%}")

--- Inference Results ---
Ground Truth: Bonafide
Prediction: Bonafide
Confidence: 100.00%
